In [11]:
from client import *
from common_imports import *

In [12]:
# ── MCP tool for the researcher ────────────────────────────────────────
async with MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
) as mcp_server:
    # ── Agents ─────────────────────────────────────────────────────────

    triage = Agent(
        client=client,
        name="triage",
        require_per_service_call_history_persistence=True,
        instructions=(
            "You are a triage coordinator for a content creation team. "
            "Analyze the user's request and hand off to the most appropriate agent: "
            "'researcher' for fact-gathering (preferred first step for most requests), "
            "'writer' for drafting, or 'editor' for review. "
            "Do NOT produce content yourself — just decide who should start and hand off."
        ),
    )

    researcher = Agent(
        client=client,
        name="researcher",
        require_per_service_call_history_persistence=True,
        instructions=(
            "You are a researcher. Use the Microsoft Learn search tool to find "
            "relevant, up-to-date documentation on the given topic. "
            "The Microsoft Agent Framework Python package is documented at "
            "learn.microsoft.com/agent-framework — search for terms like "
            "'agent framework workflow', 'agent framework orchestrations', etc. "
            "Do NOT confuse it with Microsoft Bot Framework — they are different products. "
            "Produce 3-5 concise bullet points summarizing your findings. "
            "When done, hand off to the writer."
        ),
        tools=[mcp_server],
    )

    writer = Agent(
        client=client,
        name="writer",
        require_per_service_call_history_persistence=True,
        instructions=(
            "You are a social media writer specializing in LinkedIn. "
            "Take the researcher's bullet points and write a punchy LinkedIn post "
            "(80-120 words). Use a hook opening, short paragraphs, and a clear CTA. "
            "Include 2-3 relevant hashtags at the end. When done, hand off to the editor."
        ),
    )

    editor = Agent(
        client=client,
        name="editor",
        require_per_service_call_history_persistence=True,
        instructions=(
            "You are a LinkedIn editor. Review the writer's draft for clarity, tone, and engagement. "
            "If you see issues (weak hook, filler, vague CTA, poor formatting), "
            "give 2-3 specific critiques and hand off to the writer for revision. "
            "If the draft is solid or has already been revised, output the polished version "
            "prefixed with 'FINAL:' on the first line. "
            "Each response must EITHER hand off OR output FINAL — never both."
        ),
    )

    # ── Build the handoff workflow (autonomous — no user interaction) ──

    workflow = (
        HandoffBuilder(
            name="content_pipeline",
            participants=[triage, researcher, writer, editor],
            termination_condition=lambda conversation: (
                len(conversation) > 0 and conversation[-1].text.strip().startswith("FINAL:")
            ),
        )
        .with_start_agent(triage)
        .with_autonomous_mode()
        .build()
    )


In [13]:
prompt = "Write a LinkedIn post about deploying Python agents on Azure Container Apps."

In [14]:
response = await workflow.run(prompt)

In [15]:
outputs = response.get_outputs()

final_text = next(
    msg.text
    for output in reversed(outputs)
    for msg in reversed(output.messages)
    if msg.text.strip().startswith("FINAL:")
)

print(final_text)

FINAL:  

🚀 **Deploy Python Agents Effortlessly With Azure Container Apps**  

Scalability and simplicity are within reach for your Python agent deployments! Azure Container Apps makes it 
seamless to deploy cloud-ready, containerized solutions while keeping performance and security top of mind.  

Here’s why developers are turning to Azure Container Apps:  

🔹 **Hassle-Free Containerization**: Package Python agents using Docker or Azure CLI, and unlock features like 
multi-turn conversations, webhook integrations, and real-time streaming—all optimized for cloud environments.  

🔹 **Rock-Solid Configurations**: Skip hardcoding with **Azure App Configuration**. Use YAML files to set 
parameters and manage secrets securely and efficiently.  

🔹 **Powerful CI/CD Integration**: Automate your entire deployment pipeline with **GitHub Actions** and **Azure 
Container Registry (ACR)**. Enable instant updates with ACR webhooks—deploy changes faster than ever!  

🔹 **Comprehensive Monitoring**: Azure’s **Application Insights** equips you with real-time telemetry and deep 
analytics. Identify bottlenecks, trace dependencies, and optimize performance with ease.  

💡 **Take the Leap**: Ready to transform your Python agent workflows? Explore Azure Container Apps today and 
experience deployment without limits!  

#PythonAgents #AzureContainerApps #CloudComputing #DevOpsSolutions  

---

This polished post is ready to engage an audience with clear, actionable points, and a motivating call to action.